In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [15]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_classic.retrievers.document_compressors import EmbeddingsFilter, DocumentCompressorPipeline, LLMChainExtractor
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader

In [4]:
loader = TextLoader("./data/appendix-keywords.txt", encoding="utf-8")

text_splitter = RecursiveCharacterTextSplitter(chunk_size= 300, chunk_overlap=0)
texts = loader.load_and_split(text_splitter)

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

retriever = FAISS.from_documents(
    documents = texts,
    embedding = embeddings
).as_retriever()

# 중복제거
redundant_filter = EmbeddingsRedundantFilter(
    embeddings = embeddings
    )

# 관련도 필터
relevant_filter = EmbeddingsFilter(
    embeddings = embeddings,
    similarity_threshold = 0.5
    )

# 문서 압축
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[
        redundant_filter,
        relevant_filter,
        ]
    )

compression_retriever = ContextualCompressionRetriever(
    base_compressor = pipeline_compressor,
    base_retriever = retriever
    )

In [18]:
docs = retriever.invoke(
    "Semantic Search에 대해서 알려줘."
)

print(len(docs))

compressed_docs = compression_retriever.invoke(
    "Semantic Search에 대해서 알려줘."
)

print(len(compressed_docs))

for doc in compressed_docs:
    print(doc.page_content)

4
1
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding
